In [3]:
import pandas as pd

df = pd.read_excel("D:/ml_shit/shopflow/data/Online Retail.xlsx") 

In [4]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
df['revenue'] = df['Quantity'] * df['UnitPrice']

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

features = df[(df['InvoiceDate'].dt.month < 9) & (df['Quantity'] > 0) & (df['CustomerID'].notnull())]
features.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,revenue
42481,539993,22386,JUMBO BAG PINK POLKADOT,10,2011-01-04 10:00:00,1.95,13313.0,United Kingdom,19.5
42482,539993,21499,BLUE POLKADOT WRAP,25,2011-01-04 10:00:00,0.42,13313.0,United Kingdom,10.5
42483,539993,21498,RED RETROSPOT WRAP,25,2011-01-04 10:00:00,0.42,13313.0,United Kingdom,10.5
42484,539993,22379,RECYCLING BAG RETROSPOT,5,2011-01-04 10:00:00,2.10,13313.0,United Kingdom,10.5
42485,539993,20718,RED RETROSPOT SHOPPER BAG,10,2011-01-04 10:00:00,1.25,13313.0,United Kingdom,12.5


In [6]:
target = df[(df['InvoiceDate'].dt.month == 9) & (df['InvoiceDate'].dt.year == 2011) & (df['Quantity'] > 0) & (df['CustomerID'].notnull())]

target = target[['CustomerID']].drop_duplicates()

target['target'] = 1
target.set_index('CustomerID', inplace=True)

target.head()

,target
CustomerID,
13509.0,1
13305.0,1
16187.0,1
17306.0,1
13631.0,1


In [7]:
new_target = df[(df['InvoiceDate'].dt.month < 9) & (df['InvoiceDate'].dt.year <= 2011) & (df['Quantity'] > 0) & (df['CustomerID'].notnull())]

new_target = new_target[['CustomerID']]
new_target = new_target.drop_duplicates()

new_target['target'] = 0
new_target.set_index('CustomerID', inplace=True)

full = pd.concat([new_target, target])
full = full[~full.index.duplicated(keep='last')]

full

,target
CustomerID,
13313.0,0
18097.0,0
16875.0,0
16282.0,0
13963.0,0
...,...
13021.0,1
15867.0,1
17227.0,1


In [8]:
#days since last purchase

date = features.groupby('CustomerID')['InvoiceDate'].max()
damn = abs(date - pd.to_datetime('2011-09-01'))

recency = pd.DataFrame(damn.dt.days)
recency

,InvoiceDate
CustomerID,
12346.0,225
12347.0,29
12348.0,148
12350.0,210
12352.0,162
...,...
18280.0,177
18281.0,80
18282.0,26


In [9]:
# number of orders per customer

num_orders = features.groupby('CustomerID')['InvoiceNo'].nunique()
num_orders

CustomerID
12346.0    1
12347.0    4
12348.0    2
12350.0    1
12352.0    5
          ..
18280.0    1
18281.0    1
18282.0    1
18283.0    9
18287.0    1
Name: InvoiceNo, Length: 3122, dtype: int64

In [10]:
# total spent

total_spent = features.groupby('CustomerID')['revenue'].sum()
total_spent

CustomerID
12346.0    77183.60
12347.0     2079.07
12348.0      594.44
12350.0      334.40
12352.0     1561.81
             ...   
18280.0      180.60
18281.0       80.82
18282.0      100.21
18283.0      985.77
18287.0      765.28
Name: revenue, Length: 3122, dtype: float64

In [11]:
recency['num_orders'] = num_orders
recency['total_spent'] = total_spent

full_df = recency.merge(full, on='CustomerID', how='inner')
full_df.columns = ['recency', 'orders', 'spent', 'target']

In [12]:
full_df.describe()

,recency,orders,spent,target
count,3122.000000,3122.000000,3122.00000,3122.000000
mean,81.319026,3.207880,1494.72347,0.304292
std,65.212094,4.938458,5669.40348,0.460181
min,0.000000,1.000000,2.90000,0.000000
25%,26.000000,1.000000,256.00750,0.000000
50%,66.000000,2.000000,549.16500,0.000000
75%,132.000000,4.000000,1314.01000,1.000000
max,239.000000,93.000000,167764.06000,1.000000


In [13]:
median1 = full_df[full_df['target'] == 1]
median1 = median1['recency'].median()

median0 = full_df[full_df['target'] == 0]
median0 = median0['recency'].median()

print(median1)
print(median0)

38.0
79.0


In [14]:
median1_order = full_df[full_df['target'] == 1]
median1_order = median1_order['orders'].median()

median0_order = full_df[full_df['target'] == 0]
median0_order = median0_order['orders'].median()

print(median1_order)
print(median0_order)

3.0
1.0


In [15]:
median1_spent = full_df[full_df['target'] == 1]
median1_spent = median1_spent['spent'].median()

median0_spent = full_df[full_df['target'] == 0]
median0_spent = median0_spent['spent'].median()

print(median1_spent)
print(median0_spent)

1059.88
410.655


In [16]:
matrix = full_df[['recency', 'orders', 'spent']].corr()
matrix

,recency,orders,spent
recency,1.000000,-0.340716,-0.154961
orders,-0.340716,1.000000,0.527482
spent,-0.154961,0.527482,1.000000


In [17]:
june = df[(df['InvoiceDate'] < pd.to_datetime('2011-06-01')) & (df['Quantity'] > 0) & (df['CustomerID'].notnull())]

june_recency = june.groupby('CustomerID')['InvoiceDate'].max()
june_recency = (pd.to_datetime('2011-06-01') - june_recency).dt.days

june_orders = june.groupby('CustomerID')['InvoiceNo'].nunique()

june_spent = june.groupby('CustomerID')['revenue'].sum()


In [18]:
customers = df[(df['InvoiceDate'] >= pd.to_datetime('2011-06-01')) &
               (df['InvoiceDate'] < pd.to_datetime('2011-07-01')) & 
               (df['Quantity'] > 0) &
               (df['CustomerID'].notnull())
               ]
june['target'] = 0
customers['target'] = 1

customers = pd.DataFrame(customers.groupby('CustomerID')['target'].first())
june_target = pd.DataFrame(june.groupby('CustomerID')['target'].first())

targets = pd.concat([customers, june_target])
targets = targets[~targets.index.duplicated(keep='first')]

june_recency['total_orders'] = june_orders
june_recency['total_spent'] = june_spent
june_recency['target'] = targets

final_june = june_recency
final_june.columns = ['recency', 'total_orders', 'total_spent', 'target']
final_june

CustomerID
12346.0                                                       133
12347.0                                                        54
12348.0                                                        56
12350.0                                                       118
12352.0                                                        70
                                      ...                        
18283.0                                                         8
18287.0                                                         9
total_orders    CustomerID
12346.0    1
12347.0    3
12348.0  ...
total_spent     CustomerID
12346.0    77183.60
12347.0     182...
target                      target
CustomerID        
12347.0 ...
Name: InvoiceDate, Length: 2721, dtype: object

In [19]:
def timestamp(df, prediction_date, days=30):

    date = pd.to_datetime(prediction_date) # just making sure we dont have a problem with the input 
    time = date + pd.Timedelta(days=days)

    clean_df = df[(df['Quantity'] > 0) & (df['CustomerID'].notnull())]

    #making sure the data is clean and splitting into our timestamp
    feature_df = clean_df[clean_df['InvoiceDate'] < date]

    #creates the new variables
    feature_recency = feature_df.groupby('CustomerID')['InvoiceDate'].max()
    feature_recency = (date - feature_recency).dt.days.rename('recency')
                       
    feature_orders = feature_df.groupby('CustomerID')['InvoiceNo'].nunique().rename('total_orders')
    feature_spent = feature_df.groupby('CustomerID')['revenue'].sum().rename('total_spent')
    avg_order = (round(feature_spent / feature_orders, 2)).rename('avg_order')

    feature_df['months'] = feature_df['InvoiceDate'].dt.to_period('M')
    purchased_months = feature_df.groupby('CustomerID')['months'].nunique().rename('purchased_months')

    target1 = clean_df[(clean_df['InvoiceDate'] >= date) & (clean_df['InvoiceDate'] < time)]
    feature_df['target'] = 0
    target1['target'] = 1

    target1 = pd.DataFrame(target1.groupby('CustomerID')['target'].first())
    df_target = pd.DataFrame(feature_df.groupby('CustomerID')['target'].first())

    targets = pd.concat([target1, df_target])
    targets = targets[~targets.index.duplicated(keep='first')]

    final_df = pd.concat([feature_recency, feature_orders, feature_spent, avg_order, purchased_months, targets], axis=1, join='inner')
    
    return final_df

In [20]:
june = timestamp(df, '2011-06-01')
july = timestamp(df, '2011-07-01')
august = timestamp(df, '2011-08-01')
september = timestamp(df, '2011-09-01')
octuber = timestamp(df, '2011-10-01')
november = timestamp(df, '2011-11-01')

In [21]:
june['prediction_date'] = pd.to_datetime('2011-06-01')
july['prediction_date'] = pd.to_datetime('2011-07-01')
august['prediction_date'] = pd.to_datetime('2011-08-01')
september['prediction_date'] = pd.to_datetime('2011-09-01')
octuber['prediction_date'] = pd.to_datetime('2011-10-01')
november['prediction_date'] = pd.to_datetime('2011-11-01')

In [22]:
train = pd.concat([june, july, august])
val = pd.concat([september, octuber])
test = november

X_train = train[['recency', 'total_orders', 'total_spent', 'avg_order', 'purchased_months']]
y_train = train['target']

X_val = val[['recency', 'total_orders', 'total_spent', 'avg_order', 'purchased_months']]
y_val = val['target']

X_test = november[['recency', 'total_orders', 'total_spent', 'avg_order', 'purchased_months']]
y_test = november['target']

In [23]:
import numpy as np

#X_train['total_spent'] = np.log1p(X_train['total_spent'])
#X_val['total_spent'] = np.log1p(X_val['total_spent'])

X_train['total_orders'] = np.log1p(X_train['total_orders'])
X_val['total_orders'] = np.log1p(X_val['total_orders'])

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_curve, roc_auc_score


model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_val_scaled)

print("Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification report:")
print(classification_report(y_val, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_pred))

Accuracy: 0.756526756094043

Classification report:
              precision    recall  f1-score   support

           0       0.78      0.92      0.84      4989
           1       0.62      0.34      0.44      1944

    accuracy                           0.76      6933
   macro avg       0.70      0.63      0.64      6933
weighted avg       0.74      0.76      0.73      6933


Confusion matrix:
[[4576  413]
 [1275  669]]


In [26]:
y_val_prob = model.predict_proba(X_val_scaled)[:, 1]
roc12 = roc_auc_score(y_val, y_val_prob)
roc12

0.7274164169403139

In [27]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(max_depth=6, n_estimators=15, min_samples_leaf=3, random_state=42)
forest.fit(X_train_scaled, y_train)

y_prob = forest.predict_proba(X_val_scaled)[:,1]

roc = roc_auc_score(y_val, y_prob)
roc

0.7318317891954893

In [28]:
precision, recall, thresholds = precision_recall_curve(y_val, y_prob)

pr_results = pd.DataFrame({
    'threshold': thresholds,
    'precision': precision[:-1],
    'recall': recall[:-1]
})

pr_results['f1'] = (
    2 * pr_results['precision'] * pr_results['recall']
    / (pr_results['precision'] + pr_results['recall'])
)

print(pr_results.iloc[(pr_results['threshold'] - 0.5).abs().argsort()[:1]])
print(pr_results[pr_results['threshold'].between(0.2, 0.5)].iloc[::max(1, len(pr_results[pr_results['threshold'].between(0.2, 0.5)]) // 10)])
print("\nBest F1:")
print(pr_results.loc[pr_results['f1'].idxmax()])

      threshold  precision    recall        f1
2507   0.500062    0.62782  0.343621  0.444149
      threshold  precision    recall        f1
1153   0.200193   0.408463  0.729938  0.523810
1288   0.217332   0.421299  0.693930  0.524291
1423   0.235241   0.442347  0.663066  0.530671
1558   0.252056   0.459893  0.619342  0.527839
1693   0.269544   0.481340  0.583848  0.527662
1828   0.295566   0.502134  0.544753  0.522576
1963   0.329128   0.522629  0.498971  0.510526
2098   0.378532   0.551408  0.463477  0.503633
2233   0.419462   0.572419  0.424897  0.487747
2368   0.458534   0.591195  0.386831  0.467662
2503   0.498037   0.628277  0.345165  0.445551

Best F1:
threshold    0.260518
precision    0.470775
recall       0.609053
f1           0.531061
Name: 1612, dtype: float64


In [29]:
from sklearn.ensemble import HistGradientBoostingClassifier


boost = HistGradientBoostingClassifier(max_iter=100, min_samples_leaf=81, max_depth=3)
boost.fit(X_train_scaled, y_train)

y_prob_boost = boost.predict_proba(X_val_scaled)[:,1]

roc = roc_auc_score(y_val, y_prob_boost)
roc 

0.7330094314487758

In [30]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Use the threshold selected on validation
threshold = 0.251797

# Convert test probabilities into class predictions
y_test_pred = (y_prob_boost >= threshold).astype(int)

# Evaluate on the untouched test labels
print("Threshold:", threshold)
print("Accuracy:", accuracy_score(y_val, y_test_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob_boost))
print("PR-AUC (Average Precision):", average_precision_score(y_val, y_prob_boost))

print("\nClassification Report:")
print(classification_report(y_val, y_test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_test_pred))

Threshold: 0.251797
Accuracy: 0.6904658877830665
ROC-AUC: 0.7330094314487758
PR-AUC (Average Precision): 0.5689939742037542

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.72      0.77      4989
           1       0.46      0.62      0.53      1944

    accuracy                           0.69      6933
   macro avg       0.64      0.67      0.65      6933
weighted avg       0.73      0.69      0.70      6933

Confusion Matrix:
[[3584 1405]
 [ 741 1203]]


In [31]:
from sklearn.ensemble import HistGradientBoostingClassifier


boost = HistGradientBoostingClassifier(max_iter=100, min_samples_leaf=81, max_depth=3)
boost.fit(X_train_scaled, y_train)

y_prob_boost = boost.predict_proba(X_test_scaled)[:,1]

roc = roc_auc_score(y_test, y_prob_boost)
roc 

0.73227673312936

In [32]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Use the threshold selected on validation
threshold = 0.251797

# Convert test probabilities into class predictions
y_test_pred = (y_prob_boost >= threshold).astype(int)

# Evaluate on the untouched test labels
print("Threshold:", threshold)
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_boost))
print("PR-AUC (Average Precision):", average_precision_score(y_test, y_prob_boost))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Threshold: 0.251797
Accuracy: 0.6459486663311524
ROC-AUC: 0.73227673312936
PR-AUC (Average Precision): 0.6089773305058851

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.59      0.69      2633
           1       0.48      0.75      0.59      1341

    accuracy                           0.65      3974
   macro avg       0.65      0.67      0.64      3974
weighted avg       0.71      0.65      0.66      3974

Confusion Matrix:
[[1555 1078]
 [ 329 1012]]


In [33]:
def build_features(df, prediction_date):

    date = pd.to_datetime(prediction_date) # just making sure we dont have a problem with the input 
    
    #making sure the data is clean and splited into our timestamp
    clean_df = df[(df['Quantity'] > 0) & (df['CustomerID'].notnull())]
    feature_df = clean_df[clean_df['InvoiceDate'] < date].copy()

    #creates the new variables
    feature_recency = feature_df.groupby('CustomerID')['InvoiceDate'].max()
    feature_recency = (date - feature_recency).dt.days.rename('recency')
                       
    feature_orders = feature_df.groupby('CustomerID')['InvoiceNo'].nunique().rename('total_orders')
    feature_spent = feature_df.groupby('CustomerID')['revenue'].sum().rename('total_spent')
    avg_order = (round(feature_spent / feature_orders, 2)).rename('avg_order')

    feature_df['months'] = feature_df['InvoiceDate'].dt.to_period('M')
    purchased_months = feature_df.groupby('CustomerID')['months'].nunique().rename('purchased_months')

    final_df = pd.concat([feature_recency, feature_orders, feature_spent, avg_order, purchased_months], axis=1)

    return final_df

In [34]:
def build_target(df, prediction_date, days=30):

    date = pd.to_datetime(prediction_date)
    time = date + pd.Timedelta(days=days)

    clean_df = df[
        (df['Quantity'] > 0) &
        (df['CustomerID'].notnull())
    ]

    # We can only build labels if the entire target window is observed
    if clean_df['InvoiceDate'].max() < time:
        raise ValueError("The full target window is not available.")

    # Purchases made during the target window
    target_df = clean_df[
        (clean_df['InvoiceDate'] >= date) &
        (clean_df['InvoiceDate'] < time)
    ]

    # One positive label per customer who purchased
    target1 = (
        target_df.groupby('CustomerID')
        .size()
        .rename('target')
    )

    target1[:] = 1

    # Customers with features before the prediction date
    feature_customers = clean_df[
        clean_df['InvoiceDate'] < date
    ]['CustomerID'].unique()

    # Customers who did not purchase during the window get 0
    targets = pd.Series(
        0,
        index=pd.Index(feature_customers, name='CustomerID'),
        name='target'
    )

    # Mark customers who purchased as 1
    targets.loc[targets.index.intersection(target1.index)] = 1

    return targets

In [35]:
prediction_date = '2011-10-01'

features = build_features(df, prediction_date)
target = build_target(df, prediction_date)

dataset = features.join(target, how='inner')

print(dataset.shape)
print(dataset.head())
print(dataset['target'].value_counts(normalize=True))
print(dataset.isnull().sum())

(3616, 6)
            recency  total_orders  total_spent  avg_order  purchased_months  \
CustomerID                                                                    
12346.0         255             1     77183.60   77183.60                 1   
12347.0          59             5      2790.86     558.17                 5   
12348.0           5             4      1797.24     449.31                 4   
12350.0         240             1       334.40     334.40                 1   
12352.0           2             7      2194.31     313.47                 3   

            target  
CustomerID          
12346.0          0  
12347.0          0  
12348.0          0  
12350.0          0  
12352.0          0  
target
0    0.729812
1    0.270188
Name: proportion, dtype: float64
recency             0
total_orders        0
total_spent         0
avg_order           0
purchased_months    0
target              0
dtype: int64


In [40]:
from src.modeling import create_model, THRESHOLD

boost = create_model()
boost.fit(X_train_scaled, y_train)

y_prob_boost = boost.predict_proba(X_val_scaled)[:, 1]

roc = roc_auc_score(y_val, y_prob_boost)
print(roc)

ModuleNotFoundError: No module named 'src'

In [39]:
import os

print(os.getcwd())
print(os.listdir())

d:\ml_shit\shopflow\notebooks1
['modeling.ipynb']
